# Churn: train, tune and compare gradient boosting models

*Session 10, block 2 practice. Author: course team, licence CC-BY-4.0.*

We compare a random forest, scikit-learn's `HistGradientBoostingClassifier`, XGBoost, LightGBM and CatBoost with the **same** cross-validation folds, then look at class weights, early stopping and a small randomised search. The current model from Session 8, a logistic regression, is the baseline.

Theory: [02-bagging-random-forests-and-boosting.md](../theory/02-bagging-random-forests-and-boosting.md) and [03-xgboost-lightgbm-catboost.md](../theory/03-xgboost-lightgbm-catboost.md).

> On macOS, XGBoost and LightGBM need `brew install libomp`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()
num = X.select_dtypes("number").columns.tolist()

X_cat = X.astype({c: "category" for c in cat})                  # native categories for XGBoost/LightGBM
Xc_tr, Xc_te, y_tr, y_te = train_test_split(X_cat, y, test_size=0.25, stratify=y, random_state=0)
X_str_tr, X_str_te = X.loc[Xc_tr.index], X.loc[Xc_te.index]        # string columns for CatBoost
print(Xc_tr.shape, round(y_tr.mean(), 3))

## 1. Same folds for every model

In [ ]:
import time

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

cv = StratifiedKFold(5, shuffle=True, random_state=0)
onehot = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
ordinal = ColumnTransformer([("num", "passthrough", num),
                             ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)])
models = {
    "logistic regression (current)": (make_pipeline(onehot, LogisticRegression(max_iter=1000)), Xc_tr),
    "random forest": (make_pipeline(ordinal, RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                                    random_state=0, n_jobs=-1)), Xc_tr),
    "HistGradientBoosting": (HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=8,
                                                            categorical_features="from_dtype", random_state=0), Xc_tr),
    "XGBoost": (XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, enable_categorical=True,
                              n_jobs=1), Xc_tr),
    "LightGBM": (LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, n_jobs=1, verbose=-1), Xc_tr),
    # CatBoost gets cat_features in fit (see below): scikit-learn cannot clone it with a list argument
    "CatBoost": (CatBoostClassifier(n_estimators=300, learning_rate=0.05, depth=4, verbose=0,
                                    thread_count=1, allow_writing_files=False), X_str_tr),
}
rows = {}
for name, (model, data) in models.items():
    fit_params = {"cat_features": cat} if name == "CatBoost" else {}
    res = cross_validate(model, data, y_tr, cv=cv, scoring=["roc_auc", "f1"], params=fit_params)
    rows[name] = {"roc_auc": res["test_roc_auc"].mean(), "roc_auc_sd": res["test_roc_auc"].std(),
                  "f1": res["test_f1"].mean(), "fit_s": res["fit_time"].mean()}
comparison = pd.DataFrame(rows).T.round(3)
comparison

**Task 1.** Which model would you call the winner? Is the difference to the logistic regression larger than the standard deviation across folds? (Page 4 shows a paired comparison.)

## 2. Class weights

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
rows = []
for label, kw in [("no weights", {}), ("class_weight='balanced'", {"class_weight": "balanced"})]:
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, n_jobs=1, verbose=-1, **kw).fit(Xc_tr, y_tr)
    p = m.predict(Xc_te)
    rows.append({"setting": label, "precision": precision_score(y_te, p), "recall": recall_score(y_te, p),
                 "roc_auc": roc_auc_score(y_te, m.predict_proba(Xc_te)[:, 1])})
print("XGBoost equivalent: scale_pos_weight =", round(ratio, 2))
pd.DataFrame(rows).set_index("setting").round(3)

**Task 2.** Do the same with `XGBClassifier(scale_pos_weight=ratio)` and `CatBoostClassifier(auto_class_weights="Balanced")`. Then, instead of weights, lower the decision threshold of the unweighted LightGBM until its recall matches the weighted one. Compare the precisions.

In [ ]:
# TODO (Task 2)


## 3. Early stopping on a validation split of the training data

In [ ]:
import lightgbm
import matplotlib.pyplot as plt

Xf_tr, Xf_va, yf_tr, yf_va = train_test_split(Xc_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=1)
curves = {}
for lr in [0.3, 0.1, 0.03]:
    m = LGBMClassifier(n_estimators=2000, learning_rate=lr, num_leaves=8, n_jobs=1, verbose=-1)
    m.fit(Xf_tr, yf_tr, eval_X=Xf_va, eval_y=yf_va, eval_metric="binary_logloss",  # LightGBM < 4.7: eval_set=[(Xf_va, yf_va)]
          callbacks=[lightgbm.early_stopping(100, verbose=False)])
    curves[lr] = list(m.evals_result_.values())[0]["binary_logloss"]
    print(f"learning rate {lr}: best iteration {m.best_iteration_}, "
          f"test AUC {roc_auc_score(y_te, m.predict_proba(Xc_te)[:, 1]):.3f}")
for lr, c in curves.items():
    plt.plot(c, label=f"learning rate {lr}")
plt.xlabel("boosting round")
plt.ylabel("validation log loss")
plt.legend();

**Task 3.** Why does the curve of the highest learning rate rise again after its minimum? What would happen if you used the test set as `eval_set`?

## 4. A small randomised search

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

t0 = time.time()
search = RandomizedSearchCV(
    LGBMClassifier(n_estimators=200, subsample_freq=1, n_jobs=1, verbose=-1, random_state=0),
    {"learning_rate": loguniform(0.01, 0.3), "num_leaves": randint(4, 64),
     "min_child_samples": randint(5, 100), "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.5, 0.5)},
    n_iter=20, scoring="roc_auc", cv=cv, random_state=0).fit(Xc_tr, y_tr)
print({k: round(float(v), 3) for k, v in search.best_params_.items()})
print("best CV AUC", round(search.best_score_, 3), "| test AUC",
      round(roc_auc_score(y_te, search.predict_proba(Xc_te)[:, 1]), 3), "|", round(time.time() - t0), "s")

**Task 4.** Run a similar search for XGBoost (`max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `learning_rate`) and for CatBoost (`depth`, `l2_leaf_reg`, `learning_rate`). Collect the best CV and test scores and the search time in one table with the results of Section 1.

**Task 5.** Summarise in three sentences for the retention team: which model you recommend, how much better it is than the current logistic regression, and what it costs.

In [ ]:
# TODO (Tasks 4 and 5)
